# Harness Bench

**A very small benchmark to find the harness you need.**

Choose the .venv Python kernel. You can inspect a saved campaign or start a new one below. The charts retain timeouts, technical errors and unexecuted trials. A strict success rate of 0% stays visible even when some diagnostic checks pass.

In [ ]:
from pathlib import Path
import importlib
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/local_comparison.py").is_file())
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "scripts"))
import bench
import local_comparison
import scripts.campaign_results as campaign_results
importlib.reload(campaign_results)
importlib.reload(bench)
importlib.reload(local_comparison)
print("Python kernel", sys.executable)

## Choose the experiment

Demo contains a startup task and retrieval analysis. Startup runs first for each harness. The model must read numbers, write a script and calculate a count and sum on two inputs.

Use run_new_campaign = False to inspect existing results. Set campaign_id to a saved timestamp, or leave it as None for the newest local campaign. A new campaign records your requested limits and Ollama's available runtime metadata. Actual context observations can be absent when the server unloads a model.

In [ ]:
model = "lfm3:1.7b"
harnesses = ["pi", "opencode", "droid", "dsh"]
suite = "demo"
attempts = 1
timeout_sec = 180
context_window = 32768
max_output_tokens = 8192

run_new_campaign = False  # Change to True when you want to launch.
campaign_id = None

## Prepare the notebook

This cell installs matplotlib in the selected Python environment if needed. Docker and Ollama preparation happens only when you request a new campaign. Start their applications first.

In [ ]:
import importlib.util
import json
import shutil
import subprocess
import urllib.request

if importlib.util.find_spec("matplotlib") is None:
    installer = (["uv", "pip", "install", "--python", sys.executable]
                 if shutil.which("uv") else [sys.executable, "-m", "pip", "install"])
    subprocess.run(installer + ["matplotlib>=3.9,<4"], check=True)

if run_new_campaign:
    subprocess.run(["docker", "info"], check=True, capture_output=True)
    image = subprocess.run(
        ["docker", "image", "inspect", "harness-bench-base:1"], capture_output=True
    )
    if image.returncode:
        subprocess.run(["docker", "build", "-t", "harness-bench-base:1", "docker"],
                       cwd=root, check=True)
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as response:
        installed_models = json.load(response)["models"]
    if model not in {item["name"] for item in installed_models}:
        subprocess.run(["ollama", "pull", model], check=True)

## Run or select a campaign

The progress bar counts finished trials. While a trial runs, it shows the latest available event and the time since a native log changed. Empty logs are reported as awaiting events. Interrupt the kernel to cancel. Once the campaign has been created, recoverable failures and cancellations still return its partial report path.

In [ ]:
if run_new_campaign:
    campaign = local_comparison.run_comparison(
        model=model, harnesses=harnesses, suite=suite, attempts=attempts,
        timeout_sec=timeout_sec, context_window=context_window,
        max_output_tokens=max_output_tokens,
    )
else:
    available = local_comparison.saved_campaigns()
    campaign = (root / "runs/campaigns" / campaign_id if campaign_id
                else next(iter(available), None))
if campaign is None:
    raise ValueError("No local campaign saved yet. Set run_new_campaign = True to start.")
print("Campaign", campaign)
rows = campaign_results.load_rows(campaign)

## Results in percentages

Strict success uses every scheduled trial as its denominator. The outcome chart explains zero scores. The diagnostics chart uses only trials with each check recorded. N/A marks an unavailable check. For older retrieval campaigns, saved public answers provide limited diagnostics without rerunning the agent. Hidden checks remain unavailable for those campaigns.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

figure = campaign_results.plot_results(rows)
display(figure)
plt.close(figure)
# Optional export, executed only when you uncomment these lines:
# figure.savefig(campaign / "results.png", dpi=160, bbox_inches="tight")
# figure.savefig(campaign / "results.svg", bbox_inches="tight")

## Inspect each trial

Use the logs folder to inspect native events, exception.txt, verifier output and generated programs. A timeout may still have partial files. Review the error and diagnostics before comparing harnesses.

In [ ]:
import html
import os
from IPython.display import HTML

columns = ["harness", "task", "attempt", "status", "reward", "agent_seconds"]
header = "".join("<th>" + html.escape(key) + "</th>" for key in columns)
body = []
for row in rows:
    values = "".join("<td>" + html.escape(str(row.get(key) or "unknown")) + "</td>"
                     for key in columns)
    result = Path(row["result"]) if row.get("result") else None
    if result and result.exists():
        url = os.path.relpath(result.parent, Path.cwd())
        link = '<a href="' + html.escape(url, quote=True) + '">Open trial files</a>'
    else:
        link = "No trial files"
    body.append("<tr>" + values + "<td>" + link + "</td></tr>")
display(HTML("<table><thead><tr>" + header + "<th>Logs</th></tr></thead><tbody>"
             + "".join(body) + "</tbody></table>"))
for row in rows:
    checks = json.loads(row.get("criteria_json") or "{}")
    print(row["harness"], row["task"], checks)
    if row.get("error_message"):
        print(row["error_message"])